# 01 · Baselines: base model zero-shot and few-shot

Runs **Qwen3-4B-Instruct-2507** (4-bit, no adapter) on the test set in two setups and scores them with `src/eval.py`:
- **zero-shot**: condensed labeling rules as the system prompt
- **few-shot**: same + 3 fixed train examples

Runtime: **GPU T4** (Runtime → Change runtime type). Predictions are written to Google Drive and resumed after a disconnect.

⚠️ Until the hand review is frozen (`scripts/review.py freeze` → `data/splits/test.jsonl`), the test labels are **provisional** and so are these metrics.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/Benarbiaa/email-extraction-qlora.git"
import os
if not os.path.exists("/content/email-extraction-qlora"):
    !git clone -q {REPO_URL} /content/email-extraction-qlora
%cd /content/email-extraction-qlora
!git log --oneline -1

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets jsonschema
import transformers, peft, trl, bitsandbytes, torch
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__,
      "| bnb", bitsandbytes.__version__, "| torch", torch.__version__)

## 2. Persist predictions on Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/email-extraction-qlora"
!mkdir -p {DRIVE_DIR}/preds {DRIVE_DIR}/tables
!rm -rf results/preds && mkdir -p results && ln -s {DRIVE_DIR}/preds results/preds
GOLD = "data/splits/test.jsonl" if os.path.exists("data/splits/test.jsonl") else "data/splits/test_provisional.jsonl"
print("gold:", GOLD)

## 3. Smoke test (3 emails): check the model loads and outputs JSON

In [ ]:
!python src/infer.py --mode zeroshot --split {GOLD} --limit 3 --out /tmp/smoke.jsonl
!cat /tmp/smoke.jsonl

## 4. Zero-shot baseline

In [ ]:
!python src/infer.py --mode zeroshot --split {GOLD} --out results/preds/base_zeroshot.jsonl
!python src/eval.py --pred results/preds/base_zeroshot.jsonl --gold {GOLD} --name base_zeroshot

## 5. Few-shot baseline

In [ ]:
!python src/infer.py --mode fewshot --split {GOLD} --out results/preds/base_fewshot.jsonl
!python src/eval.py --pred results/preds/base_fewshot.jsonl --gold {GOLD} --name base_fewshot

## 6. Summary (copied to Drive)

In [ ]:
import pandas as pd
!cp results/tables/summary.csv results/tables/metrics_base_*.json {DRIVE_DIR}/tables/
pd.read_csv("results/tables/summary.csv").set_index("name").T